In [1]:
%cd /content/drive/MyDrive/github/Breakout

/content/drive/MyDrive/github/Breakout


In [2]:
from pathlib import Path
import torch
import torch.nn as nn
import random
from torch.optim import AdamW
from src.models import *
from src.buffer import *
from collections import deque
import numpy as np
import gymnasium as gym
import ale_py
import time

gym.register_envs(ale_py)
print(f'Device: {device}')
ROOT = Path.cwd()
saved_path = ROOT / 'checkpoint' / 'checkpoint.tar'
env = gym.make('ALE/Breakout-v5', obs_type='ram', mode=0, difficulty=0, repeat_action_probability=0)



Device: cuda


In [3]:
# @title Main

q = QNetwork().to(device)
q_target = QNetwork().to(device)
optimizer = AdamW(q.parameters(), lr=3e-4)

try:
    if device == 'cpu':
        checkpoint = torch.load(saved_path, weights_only=False, map_location='cpu')
    else:
        checkpoint = torch.load(saved_path, weights_only=False)
    q.load_state_dict(checkpoint['model'])
    q_target.load_state_dict(checkpoint['model'])

    optimizer.load_state_dict(checkpoint['optimizer'])

    buffer = ReplayBuffer.load_state(checkpoint['state'])

    epsilon = checkpoint['epsilon']
    best_reward = checkpoint['best_reward']
    step = checkpoint['step']
    episode = checkpoint['episode']
    print('Checkpoint Loaded')
except Exception as e:
    print(e)
    buffer = ReplayBuffer()
    epsilon = 1.0
    best_reward = 0
    step, episode = 0, 0
    print('Checkpoint Not Loaded')

# Parameters
gamma = 0.99
batch_size = 128

epsilon_max = 1.0
epsilon_min = 0.05
random_episodes = 1000

target_update = 1000
print_episode = 100
reward_history = deque(maxlen=print_episode)
loss_episode_history = deque()
loss_history = deque(maxlen=print_episode)

now = time.time()

while True:
    episode += 1
    s, _ = env.reset(seed=42)
    done = False
    episode_reward = 0

    # Start One Episode
    while not done:
        if random.random() < epsilon:
            a = env.action_space.sample()
        else:
            with torch.no_grad():
                qs = q(torch.from_numpy(s).to(device))
                a = qs.argmax().item()

        s2, r, terminated, truncated, _ = env.step(a)
        done = terminated or truncated

        buffer.push(s, a, r, s2, done)

        s = s2
        episode_reward += r
        step += 1

        if len(buffer) <= batch_size:
            continue

        states, actions, rewards, next_states, dones = buffer.sample(batch_size)

        q_values = q(states).gather(1, actions.unsqueeze(1)).squeeze()  # (B, )

        with torch.no_grad():
            next_q = q_target(next_states).max(1)[0]  # (B, )
            target = rewards + gamma * next_q * (1 - dones)

        loss = nn.functional.huber_loss(q_values, target)
        loss_episode_history.append(loss.item())

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % target_update == 0:
            q_target.load_state_dict(q.state_dict())
            mean_loss = np.mean(loss_episode_history)
            loss_history.append(mean_loss)
            loss_episode_history.clear()

    reward_history.append(int(episode_reward))
    mean_reward = np.mean(reward_history)

    epsilon = max(epsilon - (epsilon_max - epsilon_min) / random_episodes, epsilon_min)

    if mean_reward > best_reward:
        best_reward = episode_reward
        print(f"New Max Reward: {best_reward:.2f}")

    if mean_reward >= 40:
        break

    if episode % print_episode == 0:
        torch.save({
            'model': q.state_dict(),
            'optimizer': optimizer.state_dict(),
            'state': buffer.state(),
            'epsilon': epsilon,
            'best_reward': best_reward,
            'step': step,
            'episode': episode
        }, saved_path)

        print(f"Ep: {episode} |",
                f"reward: {mean_reward:.2f} |",
                f"Epsilon: {epsilon:.2f} |",
                f"Loss: {np.mean(loss_history):.4f} |",
                f"time: {int(time.time() - now)}s"
            )

        now = time.time()



Checkpoint Loaded
Ep: 9200 | reward: 3.87 | Epsilon: 0.05 | Loss: 0.0054 | time: 124s


KeyboardInterrupt: 